# Wildfire Dataset Final Report
STAT 301 April 18, 2026

G30: Statistically Significant

Alyson Kosek

Daksh Mathur

Tyler Stevenson

Eunice Wong

# Introduction

Wildfires are a common type of disaster in some regions, including Canada. A wildfire is an unplanned fire in an area of combustible vegetation, and can severely impact humans and animals' lives. Wildfires can be classified by cause of ignition, physical properties, combustible material present, and the effect of weather on the fire. Wildfire severity results from a combination of factors such as available fuels, physical setting, and weather. (‘Wildfire’. Wikipedia, 10 Apr. 2026. Wikipedia, https://en.wikipedia.org/w/index.php?title=Wildfire&oldid=1348075434.)

We would like to investigate the factors that are most influential to the size of a wildfire using Alberta Historical Wildfire Dataset. This is a dataset that contains details of the wildfire incidents recorded from 2006 to 2024. We will focus on inference, and aim to understand which variable(s) (given in the dataset) is/are associated with the sizes of wildfire incidents, such as causes of ignition and weather. 

The main question that this report will answer is "which factors that are most influential to the size of wildfire?".

# Methods
## Data

In [1]:
install.packages("devtools")

also installing the dependency ‘fs’


Warning message in install.packages("devtools"):
“installation of package ‘fs’ had non-zero exit status”
Warning message in install.packages("devtools"):
“installation of package ‘devtools’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [2]:
devtools::install_github("diverse-data-hub/diversedata")

Skipping install of 'diversedata' from a github remote, the SHA1 (4d9ebacf) has not changed since last install.
  Use `force = TRUE` to force installation



In [3]:
library(diversedata)
library(tidyverse)
library(glmnet)
library(broom)
library(repr)
library(dplyr)
library(ggplot2)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glmnet 4.1-8



In [4]:
# To load a data set into the environment:
data("wildfire")
head(wildfire)

year,fire_number,current_size,size_class,latitude,longitude,fire_origin,general_cause,responsible_group,activity_class,⋯,ia_arrival_at_fire_date,ia_access,fire_fighting_start_date,fire_fighting_start_size,bucketing_on_fire,first_bh_date,first_bh_size,first_uc_date,first_uc_size,first_ex_size_perimeter
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dttm>,<chr>,<dttm>,<dbl>,<chr>,<dttm>,<dbl>,<dttm>,<dbl>,<dbl>
2006,PWF001,0.10,A,56.25,-117.18,Land Owner,Resident,Resident,Grass,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-02 22:00:00,0.01,2006-04-02 22:00:00,0.01,0.10
2006,EWF002,0.20,B,53.61,-115.92,Fire Department,Incendiary,Others,Lighting Fires,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-03 13:20:00,0.20,2006-04-03 13:20:00,0.20,0.20
2006,EWF001,0.50,B,53.61,-115.59,Fire Department,Incendiary,Others,Lighting Fires,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-03 13:23:00,0.50,2006-04-03 13:23:00,0.50,0.50
2006,EWF003,0.01,A,53.61,-115.61,Industry,Incendiary,Others,Lighting Fires,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-03 14:08:00,0.01,2006-04-03 14:08:00,0.01,0.01
2006,PWF002,0.10,A,56.25,-117.05,Fire Department,Other Industry,Employees,Refuse,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-03 19:57:00,0.10,2006-04-03 20:19:00,0.10,0.10
2006,CWF001,0.20,B,51.15,-115.03,Fire Department,Resident,Resident,Unclassified,⋯,NA,Unknown,NA,0.02,Unknown,2006-04-02 16:00:01,0.20,2006-04-02 16:00:01,0.20,0.20


The dataset is called Wildfire, sourced from the Government of Alberta (https://diverse-data-hub.github.io/). Its data collection is fully observational, and contains information about the wildfires in Alberta, Canada. The dataset includes 26551 observations and 35 variables of which contain a range of types, including numeric, date, and character. 

Numeric variables include: 'year', 'current_size', 'latitude', 'longitude', 'assessment_hectares', 'fire_spread_rate', 'temperature', 'relative_humidity', 'wind_speed', 'fire_fighting_start_size', 'first_bh_size', 'first_uc_size', first_ex_size_perimeter'.

Categorical variables include: 'size_class', 'fire_origin', 'general_cause', 'responsible_group', 'activity_class', 'true_cause', 'detection_agent_type', 'detection_agent', fire_type, fire_position_on_slope, weather_conditions_over_fire, 'wind_direction', 'fuel_type', 'initial_action_by', 'ia_access', 'bucketing_on_fire'.

Date variables include: 'fire_start_date', 'ia_arrival_at_fire_date', 'fire_fighting_start_date', 'first_bh_date', 'first_uc_date'.

More information of the variable types is output from the code below:

In [5]:
glimpse(wildfire)

Rows: 26,551
Columns: 35
$ year                         <dbl> 2006, 2006, 2006, 2006, 2006, 2006, 2006,…
$ fire_number                  <chr> "PWF001", "EWF002", "EWF001", "EWF003", "…
$ current_size                 <dbl> 0.10, 0.20, 0.50, 0.01, 0.10, 0.20, 0.01,…
$ size_class                   <chr> "A", "B", "B", "A", "A", "B", "A", "A", "…
$ latitude                     <dbl> 56.25, 53.61, 53.61, 53.61, 56.25, 51.15,…
$ longitude                    <dbl> -117.18, -115.92, -115.59, -115.61, -117.…
$ fire_origin                  <chr> "Land Owner", "Fire Department", "Fire De…
$ general_cause                <chr> "Resident", "Incendiary", "Incendiary", "…
$ responsible_group            <chr> "Resident", "Others", "Others", "Others",…
$ activity_class               <chr> "Grass", "Lighting Fires", "Lighting Fire…
$ true_cause                   <chr> "Permit Related", "Arson Suspected", "Ars…
$ fire_start_date              <dttm> 2006-04-02 12:00:00, 2006-04-03 12:10:00…
$ detection_age

As our goal is to find out the relationships between wildfire variables and the size of fire, we selected the response variable to be 'current_size'. We suspect that factors related to the weather will be most impactful to the spread of wildfires, such as 'weather_conditions_over_fire', 'temperature', 'relative_humidity', 'wind_direction' and 'wind_speed'. We think that other variables like investigation documents of the wildfire cases seems to be of less relevance to the size of wildfires.

Since there are 35 columns/variables in the dataset, we decided to drop a subset that would introduce data leakage as these variables contain information not available at prediction time or directly contain the outcome. Keeping these variables would make the model look better than it really is and hurt how well it works on new data.

First, we will drop 'size_class', 'fire_fighting_start_size', 'first_bh_size', 'first_uc_size', 'assessment_hectares', and 'first_ex_size_perimeter', as these variables are measurements of the wildfire size at different times, and will cause data leakage during our model training.

We will extract the months of the wildfire incidents to find out if there is relavance between the months/seasons and fire spread. Afterwards, we will drop the date variables as they are redundant and difficult to include in our model training, and time to control a fire can also leak information of the fire size: 'first_bh_date', 'fire_fighting_start_date', 'first_uc_date', 'ia_arrival_at_fire_date'.

We will also drop variables of detection agent or fire-fighting agents as these variables are also collected after the inital assement and will cause data leakage: 'detection_agent_type', 'detection_agent', 'initial_action_by', 'ia_access', and 'bucketing_on_fire'.

Lastly, we will drop the 'fire_number' column, as it is the IDs of each wildfire case.


After dropping those columns, we found that 'temperature', 'relative_humidity', 'wind_speed' columns have some missing values. As we have a large dataset, we will just drop those rows with missing values as we still have a lot of data to train our model with.

# References

1. ‘Wildfire’. Wikipedia, 10 Apr. 2026. Wikipedia, https://en.wikipedia.org/w/index.php?title=Wildfire&oldid=1348075434.
2. Burak, K., Khoda, E. E., Piran, A., Ramirez, F., Subrahmanian, S., and Ta, S. (2025). diversedata: Diverse Data Hub. R Package Version 1.0.0, https://github.com/diverse-data-hub/diversedata